# Deep-CASAS: Activity Recognition with LSTM
Training on **kyoto7** and **kyoto11** datasets using a 3-fold cross-validated LSTM.

## 1. Install Dependencies

In [ ]:
!pip install -q tensorflow scikit-learn numpy

## 2. Mount Google Drive and Set Up Files
Upload your `dataset/kyoto7` and `dataset/kyoto11` raw files to Google Drive, then mount below.
Alternatively, run the **Upload** cell to upload directly.

In [ ]:
# Option A: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# After mounting, copy your project folder to /content/
# Adjust the path below to where you placed the project on Drive
DRIVE_PROJECT_PATH = '/content/drive/MyDrive/deep-casas'  # <-- change if needed

import os, shutil
if os.path.exists(DRIVE_PROJECT_PATH):
    # Copy dataset folder to /content/dataset
    if not os.path.exists('/content/dataset'):
        shutil.copytree(os.path.join(DRIVE_PROJECT_PATH, 'dataset'), '/content/dataset')
    # Copy scripts
    for f in ['data.py', 'models.py', 'train.py']:
        src = os.path.join(DRIVE_PROJECT_PATH, f)
        if os.path.exists(src):
            shutil.copy(src, f'/content/{f}')
    print('Files copied from Drive.')
else:
    print(f'Path not found: {DRIVE_PROJECT_PATH}  -- use Option B below instead.')

In [ ]:
# Option B: Upload files manually (run this if NOT using Drive)
# Upload data.py, models.py, train.py and the raw dataset files (kyoto7, kyoto11)
from google.colab import files
import os

print('Upload data.py, models.py, train.py, and the raw kyoto7 / kyoto11 dataset files.')
uploaded = files.upload()

# Move dataset files into /content/dataset/
os.makedirs('/content/dataset', exist_ok=True)
for name in ['kyoto7', 'kyoto11']:
    if name in uploaded:
        with open(f'/content/dataset/{name}', 'wb') as f:
            f.write(uploaded[name])
        print(f'Saved dataset/{name}')

## 3. Verify Scripts and Dataset

In [ ]:
import os

required_scripts = ['/content/data.py', '/content/models.py', '/content/train.py']
required_data    = ['/content/dataset/kyoto7', '/content/dataset/kyoto11']

print('--- Scripts ---')
for p in required_scripts:
    status = '✓' if os.path.exists(p) else '✗  MISSING'
    print(f'  {status}  {p}')

print('--- Dataset files ---')
for p in required_data:
    if os.path.exists(p):
        size_mb = os.path.getsize(p) / 1e6
        print(f'  ✓  {p}  ({size_mb:.1f} MB)')
    else:
        print(f'  ✗  MISSING  {p}')

## 4. Preprocess Datasets → Save .npy Files
> **If you already ran this before:** delete the old `.npy` files first (`!rm -rf /content/npy`) so they are regenerated with the correct **post-padding** (required for cuDNN LSTM on GPU).

In [ ]:
%cd /content
# Remove old npy files to force regeneration with post-padding
!rm -rf /content/npy
!python data.py

In [ ]:
# Verify .npy files were created
!ls -lh /content/npy/

## 5. Train the Model
Available models: `LSTM`, `biLSTM`, `CascadeLSTM`, `Ensemble2LSTM`, `CascadeEnsembleLSTM`

In [ ]:
MODEL = 'LSTM'   # change to biLSTM / CascadeLSTM / Ensemble2LSTM / CascadeEnsembleLSTM

%cd /content
!python train.py --v {MODEL} 2>&1 | tee train_output.log

## 6. Review Results

In [ ]:
# Print the last 100 lines of the training log
!tail -100 /content/train_output.log

In [ ]:
# List generated CSV logs and saved model checkpoints
import glob
print('--- CSV training logs ---')
for f in sorted(glob.glob('/content/*.csv')):
    print(' ', f)

print('--- Saved model checkpoints ---')
for f in sorted(glob.glob('/content/*.keras') + glob.glob('/content/*.h5')):
    print(' ', f)

In [ ]:
# (Optional) Plot training accuracy from a CSV log
import glob, pandas as pd, matplotlib.pyplot as plt

logs = sorted(glob.glob('/content/LSTM-*.csv') + glob.glob('/content/biLSTM-*.csv') +
              glob.glob('/content/CascadeLSTM-*.csv'))
if not logs:
    print('No epoch-level CSV logs found yet.')
else:
    for log_path in logs:
        try:
            df = pd.read_csv(log_path)
            if 'accuracy' in df.columns:
                plt.figure(figsize=(10, 4))
                plt.plot(df['epoch'], df['accuracy'], label='train acc')
                if 'val_accuracy' in df.columns:
                    plt.plot(df['epoch'], df['val_accuracy'], label='val acc')
                plt.title(os.path.basename(log_path))
                plt.xlabel('Epoch'); plt.ylabel('Accuracy')
                plt.legend(); plt.tight_layout(); plt.show()
        except Exception as e:
            print(f'{log_path}: {e}')

## 7. (Optional) Download Results

In [ ]:
from google.colab import files
import glob

# Download all CSV score files
for f in glob.glob('/content/cv-scores-*.csv'):
    files.download(f)

# Download training logs
for f in glob.glob('/content/*.csv'):
    if not f.startswith('/content/cv-scores'):
        files.download(f)